GT4Py - GridTools Framework

Copyright (c) 2014-2024, ETH Zurich
All rights reserved.

Please, refer to the LICENSE file in the root directory.
SPDX-License-Identifier: BSD-3-Clause

# Shallow Water Model

A doubly periodic shallow water model on an Arakawa C-grid, integrated with a leapfrog
scheme and a Robert-Asselin time filter. It is the NCAR
[SWM](https://github.com/NCAR/SWM) mini-app, a long-standing benchmark for comparing
programming models, written here as a single `gt4py.next` field operator.

The prognostic equations are

$$
\frac{\partial \mathbf{V}}{\partial t} + (\zeta + f)\,\mathbf{k} \times \mathbf{V}
  + \nabla\left(P + \tfrac{1}{2}\mathbf{V}\cdot\mathbf{V}\right) = 0,
\qquad
\frac{\partial P}{\partial t} + \nabla\cdot(P\mathbf{V}) = 0 ,
$$

discretised with the potential-enstrophy-conserving finite differences of Sadourny (1975,
his Eq. 4). The four intermediates below are his mass fluxes and diagnostics,

$$
U = \overline{P}^{\,x} u, \qquad V = \overline{P}^{\,y} v, \qquad
H = P + \tfrac{1}{2}\left(\overline{u^2}^{\,x} + \overline{v^2}^{\,y}\right), \qquad
\eta = \frac{\delta_x v - \delta_y u}{\overline{P}^{\,xy}} ,
$$

named `cu`, `cv`, `h` and `z` here.

**Provenance.** The *scheme* is Sadourny's. The *configuration* --- grid size, spacing,
time step, filter coefficient and initial condition --- comes from the
[NCAR/SWM](https://github.com/NCAR/SWM) benchmark rather than from the paper; Sadourny
damps the leapfrog scheme by averaging odd and even time levels every $N$ steps, not with
the per-step Robert-Asselin filter used here.

In [ ]:
import numpy as np

import gt4py.next as gtx
from gt4py.next.experimental import concat_where

import swm_numpy

## Grid and backend

Fields carry **one halo cell on every side**, so the domain runs from `-1` to `M+1`. All
three prognostic fields share this one index space; the C-grid staggering is expressed by
the shifts inside the operators rather than by different array shapes.

In [ ]:
I = gtx.Dimension("I")
J = gtx.Dimension("J")

IJField = gtx.Field[gtx.Dims[I, J], gtx.float64]

M = swm_numpy.M  # 16
N = swm_numpy.N  # 16

DX = swm_numpy.DX  # 100 km
DY = swm_numpy.DY
DT = swm_numpy.DT  # 90 s
ALPHA = swm_numpy.ALPHA  # Robert-Asselin filter coefficient

backend = None
# backend = gtx.gtfn_cpu
# backend = gtx.gtfn_gpu

## Differential operators

Each operator comes in two variants that differ only in shift direction. The forward form
($+1$) maps an unstaggered quantity onto a staggered location, the backward form ($-1$)
maps back. Which one a term needs is fixed by where its operands live: `p` and `h` sit at
cell centres, `u` and `cu` on $x$-faces, `v` and `cv` on $y$-faces, and `z` on corners.

Keeping the two variants apart by hand is exactly the bookkeeping that
[staggered dimensions](../docs/development/ADRs/next/0026-Staggered_Dimensions.md) are
designed to remove; a follow-up rewrites these eight operators as four.

In [ ]:
@gtx.field_operator
def avg_x(f: IJField) -> IJField:
    """Average onto the next x-face."""
    return 0.5 * (f(I + 1) + f)


@gtx.field_operator
def avg_y(f: IJField) -> IJField:
    """Average onto the next y-face."""
    return 0.5 * (f(J + 1) + f)


@gtx.field_operator
def avg_x_staggered(f: IJField) -> IJField:
    """Average an x-staggered field back onto cell centres."""
    return 0.5 * (f(I - 1) + f)


@gtx.field_operator
def avg_y_staggered(f: IJField) -> IJField:
    """Average a y-staggered field back onto cell centres."""
    return 0.5 * (f(J - 1) + f)


@gtx.field_operator
def delta_x(dx: gtx.float64, f: IJField) -> IJField:
    """Forward difference in x."""
    return (1.0 / dx) * (f(I + 1) - f)


@gtx.field_operator
def delta_y(dy: gtx.float64, f: IJField) -> IJField:
    """Forward difference in y."""
    return (1.0 / dy) * (f(J + 1) - f)


@gtx.field_operator
def delta_x_staggered(dx: gtx.float64, f: IJField) -> IJField:
    """Backward difference in x, for an x-staggered field."""
    return (1.0 / dx) * (f - f(I - 1))


@gtx.field_operator
def delta_y_staggered(dy: gtx.float64, f: IJField) -> IJField:
    """Backward difference in y, for a y-staggered field."""
    return (1.0 / dy) * (f - f(J - 1))

## Periodicity

The domain wraps in both directions, and the wrap is expressed **inside** the DSL:
`concat_where` selects a sub-domain and takes its values from the opposite edge. At
$i = -1$ the field is read at $i + M$, at $i = M$ at $i - M$, and likewise in $j$.

Each `concat_where` rebinds `f`, so the $j$ pass sees the already-$i$-corrected field
and the four corners come out right. Writing the four as one simultaneous expression
would leave them wrong.

Note that the shift distance is `m`, a runtime argument rather than a literal. Domain
inference resolves the composition: at $i = -1$ the result comes from $i = M-1$, so the
interior is all that is ever demanded of the operator that produced it.

In [ ]:
@gtx.field_operator
def make_periodic(f: IJField, m: gtx.int32, n: gtx.int32) -> IJField:
    """Fill the one-cell halo of `f` from the opposite edge."""
    f = concat_where(I == -1, f(I + m), f)
    f = concat_where(I == m, f(I - m), f)
    f = concat_where(J == -1, f(J + n), f)
    f = concat_where(J == n, f(J - n), f)
    return f

## The timestep

One fused field operator produces all six updated fields. The first three are the leapfrog
update of the prognostic variables; the last three are the time-filtered previous level,

$$ u^{\text{old}} \leftarrow u + \alpha\,(u^{\text{new}} - 2u + u^{\text{old}}) , $$

which suppresses the computational mode of the leapfrog scheme.

In [ ]:
@gtx.field_operator
def timestep(
    u: IJField,
    v: IJField,
    p: IJField,
    uold: IJField,
    vold: IJField,
    pold: IJField,
    dx: gtx.float64,
    dy: gtx.float64,
    tdt: gtx.float64,
    alpha: gtx.float64,
    m: gtx.int32,
    n: gtx.int32,
) -> tuple[IJField, IJField, IJField, IJField, IJField, IJField]:
    """Advance one leapfrog step and return (unew, vnew, pnew, uold, vold, pold)."""
    cu = avg_x(p) * u
    cv = avg_y(p) * v
    z = (delta_x(dx, v) - delta_y(dy, u)) / avg_x(avg_y(p))
    h = p + 0.5 * (avg_x_staggered(u * u) + avg_y_staggered(v * v))

    unew = uold + avg_y_staggered(z) * avg_y_staggered(avg_x(cv)) * tdt - delta_x(dx, h) * tdt
    vnew = vold - avg_x_staggered(z) * avg_x_staggered(avg_y(cu)) * tdt - delta_y(dy, h) * tdt
    pnew = pold - delta_x_staggered(dx, cu) * tdt - delta_y_staggered(dy, cv) * tdt

    uold_new = u + alpha * (unew - 2.0 * u + uold)
    vold_new = v + alpha * (vnew - 2.0 * v + vold)
    pold_new = p + alpha * (pnew - 2.0 * p + pold)

    return (
        make_periodic(unew, m, n),
        make_periodic(vnew, m, n),
        make_periodic(pnew, m, n),
        uold_new,
        vold_new,
        pold_new,
    )

The prognostic fields are written over the whole domain, halo included, while the
filtered levels only need the interior. A `program` lets each output declare its own
domain.

In [ ]:
@gtx.program
def timestep_program(
    u: IJField,
    v: IJField,
    p: IJField,
    uold: IJField,
    vold: IJField,
    pold: IJField,
    dx: gtx.float64,
    dy: gtx.float64,
    tdt: gtx.float64,
    alpha: gtx.float64,
    unew: IJField,
    vnew: IJField,
    pnew: IJField,
    m: gtx.int32,
    n: gtx.int32,
):
    timestep(
        u,
        v,
        p,
        uold,
        vold,
        pold,
        dx,
        dy,
        tdt,
        alpha,
        m,
        n,
        out=(unew, vnew, pnew, uold, vold, pold),
        domain=(
            {I: (-1, m + 1), J: (-1, n + 1)},
            {I: (-1, m + 1), J: (-1, n + 1)},
            {I: (-1, m + 1), J: (-1, n + 1)},
            {I: (0, m), J: (0, n)},
            {I: (0, m), J: (0, n)},
            {I: (0, m), J: (0, n)},
        ),
    )

## Time loop

`uold`, `vold` and `pold` are updated in place and never swapped; only the prognostic
fields ping-pong with their `new` counterparts. The first cycle is a forward Euler step
with the filter switched off, which is how the leapfrog scheme is started.

Nothing here touches the halo any more --- it is filled inside the timestep.

In [ ]:
def run(itmax: int, m: int = M, n: int = N) -> tuple[gtx.Field, gtx.Field, gtx.Field]:
    """Integrate `itmax` steps and return the final `u`, `v`, `p` fields."""
    domain = gtx.domain({I: (-1, m + 1), J: (-1, n + 1)})

    u0, v0, p0 = swm_numpy.initial_conditions(m, n, DX, DY)

    def as_halo_field(a: np.ndarray) -> gtx.Field:
        return gtx.as_field(domain, np.pad(a, 1, mode="wrap"), allocator=backend)

    u, v, p = as_halo_field(u0), as_halo_field(v0), as_halo_field(p0)
    uold, vold, pold = as_halo_field(u0), as_halo_field(v0), as_halo_field(p0)
    unew, vnew, pnew = (gtx.zeros(domain, dtype=gtx.float64, allocator=backend) for _ in range(3))

    program = timestep_program.with_backend(backend) if backend is not None else timestep_program

    for cycle in range(itmax):
        tdt = DT if cycle == 0 else 2.0 * DT
        alpha = 0.0 if cycle == 0 else ALPHA

        program(
            u,
            v,
            p,
            uold,
            vold,
            pold,
            DX,
            DY,
            tdt,
            alpha,
            unew,
            vnew,
            pnew,
            m,
            n,
            offset_provider={},
        )

        u, unew = unew, u
        v, vnew = vnew, v
        p, pnew = pnew, p

    return u, v, p


def interior_of(field: gtx.Field, m: int = M, n: int = N) -> np.ndarray:
    return field.asnumpy()[1 : m + 1, 1 : n + 1]

## Validation against NumPy

The companion module `swm_numpy.py` implements the same scheme with `numpy.roll`, which
gives periodicity for free and needs no halo at all. Agreement between the two is a check
that the halo bookkeeping here is right.

In [ ]:
STEPS = 100

# The two implementations sum the same terms in a different order, so they agree to
# rounding rather than bit-for-bit. Tolerances are absolute because u and v cross zero.
ATOL_UV = 1e-11  # peak |u|, |v| ~ 4
ATOL_P = 1e-8  # peak |p| ~ 5e4


def test_matches_numpy():
    u, v, p = run(STEPS)
    u_ref, v_ref, p_ref = swm_numpy.run(STEPS)

    np.testing.assert_allclose(interior_of(u), u_ref, atol=ATOL_UV)
    np.testing.assert_allclose(interior_of(v), v_ref, atol=ATOL_UV)
    np.testing.assert_allclose(interior_of(p), p_ref, atol=ATOL_P)


test_matches_numpy()
print("GT4Py and NumPy agree after", STEPS, "steps")

## Validation against the NCAR reference

`swm_reference.npz` holds `u`, `v`, `p` after the full 4000-step benchmark, taken from
[NCAR/SWM](https://github.com/NCAR/SWM) `ref/16x16` (Apache-2.0). Reproducing it end to
end is the real correctness check, but it takes far longer than a notebook should in CI,
so it is opt-in.

In [ ]:
RUN_FULL_BENCHMARK = False


def test_matches_reference():
    u, v, p = run(swm_numpy.ITMAX)
    u_out, v_out, p_out = swm_numpy.to_reference_layout(
        interior_of(u), interior_of(v), interior_of(p)
    )

    reference = np.load("swm_reference.npz")
    np.testing.assert_allclose(u_out, reference["u"], atol=ATOL_UV)
    np.testing.assert_allclose(v_out, reference["v"], atol=ATOL_UV)
    np.testing.assert_allclose(p_out, reference["p"], atol=ATOL_P)


if RUN_FULL_BENCHMARK:
    test_matches_reference()
    print("GT4Py reproduces the NCAR reference after", swm_numpy.ITMAX, "steps")

## The flow

The initial stream function sets up a pair of counter-rotating vortices; after a few
hundred steps the pressure field has developed the finer structure the scheme is meant to
handle without spurious accumulation of energy at the grid scale.

In [ ]:
import matplotlib.pyplot as plt

u, v, p = run(STEPS)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, field, name in zip(axes, (u, v, p), ("u", "v", "p")):
    image = ax.contourf(interior_of(field).T, levels=20)
    ax.set_title(f"{name} after {STEPS} steps")
    ax.set_aspect("equal")
    fig.colorbar(image, ax=ax)
fig.tight_layout()

## References

Sadourny, R. (1975). The dynamics of finite-difference models of the shallow-water
equations. *Journal of the Atmospheric Sciences*, 32(4), 680-689.
[doi:10.1175/1520-0469(1975)032&lt;0680:TDOFDM&gt;2.0.CO;2](https://doi.org/10.1175/1520-0469(1975)032%3C0680:TDOFDM%3E2.0.CO;2)

The benchmark itself --- its configuration, initial condition and reference data --- is
[NCAR/SWM](https://github.com/NCAR/SWM), which implements the same model across a range of
languages and programming models.